## Criteria evaluator

### Install libraries

In [1]:
! pip install --upgrade langsmith langchain langchain-core langchain_openai langchain_classic langchain-text-splitters

In [2]:
from langsmith import traceable
from langchain_classic.evaluation import load_evaluator, Criteria
from langsmith import Client
from dotenv import load_dotenv
from langchain_openai import ChatOpenAI
import json
import uuid
import os

load_dotenv()

True

### Import libraries and configure LangSmith

In [3]:
# Enable LangSmith tracking (requires LangSmith account):
os.environ["LANGSMITH_TRACING"] = "true"
os.environ["LANGSMITH_PROJECT"] = "course-demo"
os.environ["LANGSMITH_ENDPOINT"] = "https://api.smith.langchain.com"
# os.environ["LANGSMITH_API_KEY"] = "<YOUR_KEY>" #included in .env

### Wygenerowanie odpowiedzi LLM

In [4]:
def message_to_text(message) -> str:
    """Converts AIMessage/dict/str to plain text writable in LangSmith."""
    content = getattr(message, "content", message)

    if isinstance(content, str):
        return content

    if isinstance(content, list):
        chunks = []
        for item in content:
            if isinstance(item, dict):
                chunks.append(str(item.get("text") or item.get("content") or item))
            else:
                chunks.append(str(item))
        return "\n".join(chunks)

    if isinstance(content, dict):
        return json.dumps(content, ensure_ascii=False)

    return str(content)

In [5]:
client = Client()

dataset_inputs = [
    "Why people don't have 3 legs?",
    "Why people are not flying?",
]

llm_test = ChatOpenAI(model="gpt-5.4", temperature=0.1, max_tokens=2048)
llm_gen = ChatOpenAI(model="gpt-5-mini", temperature=0.1, max_tokens=2048)

dataset_outputs = [
    {"answer": message_to_text(llm_test.invoke(question))}
    for question in dataset_inputs
]

print(json.dumps(dataset_outputs, indent=2, ensure_ascii=False))


[
  {
    "answer": "Humans don’t have 3 legs mainly because of **how evolution and body plans work**.\n\n### Short answer\nHumans inherited a **two-sided body plan** from very ancient ancestors:  \n- 2 arms  \n- 2 legs  \n- left/right symmetry\n\nThat basic layout is shared by almost all vertebrates (fish, amphibians, reptiles, birds, mammals).\n\n### Why not 3?\nA third leg would be unusual because:\n\n1. **Bodies are built symmetrically**  \n   Animal development usually follows a left-right pattern. Two legs fit that pattern naturally; three do not.\n\n2. **Evolution works by modifying what already exists**  \n   It doesn’t design from scratch. Since our ancestors already had four limbs, humans ended up with two arms and two legs, not an odd number.\n\n3. **Extra limbs would require major changes**  \n   A third leg would need:\n   - new bones and joints\n   - different muscles\n   - changed spine and pelvis\n   - altered nerves and blood supply\n   - a new walking pattern  \n   Th

In [6]:
eval_llm = ChatOpenAI(model="gpt-5.4", temperature=0, max_tokens=2048)

def make_langchain_string_evaluator(key: str, evaluator):
    """Adapter for LangChain Classic evaluators"""
    def _evaluate(inputs: dict, outputs: dict, reference_outputs: dict | None = None) -> dict:
        kwargs = {
            "prediction": message_to_text(outputs.get("answer", outputs)),
            "input": message_to_text(inputs.get("question", inputs)),
        }

        if reference_outputs:
            kwargs["reference"] = message_to_text(reference_outputs.get("answer", reference_outputs))

        result = evaluator.evaluate_strings(**kwargs)

        return {
            "key": key,
            "score": result.get("score"),
            "value": result.get("value"),
            "comment": result.get("reasoning") or result.get("comment") or "",
        }

    _evaluate.__name__ = key
    return _evaluate

correctness_evaluator = make_langchain_string_evaluator(
    "correctness", load_evaluator("labeled_criteria", llm=eval_llm, criteria="correctness"),
)

criteria_evaluators = [
    make_langchain_string_evaluator("helpfulness",load_evaluator("criteria", llm=eval_llm, criteria=Criteria.HELPFULNESS)),
    make_langchain_string_evaluator("relevance", load_evaluator("criteria", llm=eval_llm, criteria=Criteria.RELEVANCE)),
    make_langchain_string_evaluator("coherence",load_evaluator("criteria", llm=eval_llm, criteria=Criteria.COHERENCE)),
    make_langchain_string_evaluator("conciseness", load_evaluator("criteria", llm=eval_llm, criteria=Criteria.CONCISENESS)),
    make_langchain_string_evaluator("harmfulness", load_evaluator("criteria", llm=eval_llm, criteria=Criteria.HARMFULNESS)),
    make_langchain_string_evaluator("maliciousness", load_evaluator("criteria", llm=eval_llm, criteria=Criteria.MALICIOUSNESS)),
]

evaluators = [
    correctness_evaluator,
    *criteria_evaluators,
]


### Launch and integration with LangSmith

In [7]:
dataset_name = f"existential questions run:{uuid.uuid4()}"

dataset = client.create_dataset(
    dataset_name=dataset_name,
    description="evaluate LLM output",
)

examples = [
    {
        "inputs": {"question": question},
        "outputs": reference_output,
    }
    for question, reference_output in zip(dataset_inputs, dataset_outputs)
]

client.create_examples(dataset_id=dataset.id, examples=examples)

print(f"Created dataset: {dataset_name}")


Created dataset: existential questions run:a9bea8bb-31a8-4900-8f59-cc6b54466cd1


In [8]:
@traceable(name="generate_answer")
def generate_answer(inputs: dict) -> dict:
    """Function/chain evaluated by LangSmith."""
    response = llm_gen.invoke(inputs["question"])
    return {"answer": message_to_text(response)}

results = client.evaluate(
    generate_answer,
    data=dataset_name,
    evaluators=evaluators,
    experiment_prefix=dataset_name,
    max_concurrency=2,
)

print(results)


View the evaluation results for experiment: 'existential questions run:a9bea8bb-31a8-4900-8f59-cc6b54466cd1-e778fac8' at:
https://smith.langchain.com/o/3e1f981e-76ef-5491-9a42-e33f3bdfeba4/datasets/1234cc3b-a6c8-43a8-95e3-e7b5f77a67ee/compare?selectedSessions=37201ffa-a9f5-41c4-b4a1-f865d6956ae8




0it [00:00, ?it/s]

/tmp/ipykernel_36554/1650530555.py:14: UserWarning: Ignoring reference in CriteriaEvalChain, as it is not expected.
To use references, use the labeled_criteria instead.
  result = evaluator.evaluate_strings(**kwargs)
/tmp/ipykernel_36554/1650530555.py:14: UserWarning: Ignoring reference in CriteriaEvalChain, as it is not expected.
To use references, use the labeled_criteria instead.
  result = evaluator.evaluate_strings(**kwargs)
/tmp/ipykernel_36554/1650530555.py:14: UserWarning: Ignoring reference in CriteriaEvalChain, as it is not expected.
To use references, use the labeled_criteria instead.
  result = evaluator.evaluate_strings(**kwargs)
/tmp/ipykernel_36554/1650530555.py:14: UserWarning: Ignoring reference in CriteriaEvalChain, as it is not expected.
To use references, use the labeled_criteria instead.
  result = evaluator.evaluate_strings(**kwargs)
/tmp/ipykernel_36554/1650530555.py:14: UserWarning: Ignoring reference in CriteriaEvalChain, as it is not expected.
To use reference

<ExperimentResults existential questions run:a9bea8bb-31a8-4900-8f59-cc6b54466cd1-e778fac8>
